# Attention Mechanism


Different types of attention: 
* self attention
* multiheaded attention
* flash attention
* diffused attention

In [2]:
import numpy as np
import math, os, re 

In [4]:
L, d_k, d_v = 4, 8, 8

Q = np.random.randn(L, d_k)
K = np.random.randn(L, d_k)
V = np.random.randn(L, d_v)

Q, K, V

(array([[-0.6379047 , -0.07158755,  0.68876849,  0.43120127, -0.18231434,
         -0.2383763 , -1.07006896,  1.44503946],
        [-1.1100645 , -1.53948277, -0.35442635,  1.80797522,  0.71114946,
         -0.95961456,  1.78465927,  0.45660079],
        [ 0.82126052,  0.20626786, -0.65357876, -0.37342271,  1.20564816,
          2.87308758,  0.85419281,  0.67358072],
        [-0.24600006, -1.51042967,  0.69549591,  0.59268144,  0.82845408,
          0.3836908 , -0.7293433 ,  0.93496975]]),
 array([[-0.34542868,  0.66476068, -0.16672306, -0.92377177,  0.06217624,
         -0.67600455, -1.0378557 ,  2.32660481],
        [ 0.18577517,  0.61545414, -0.52034246,  1.06307549, -0.11730979,
          0.5289909 ,  2.42287746, -0.45666121],
        [ 2.08545483, -1.08142644, -0.78928325,  0.01764214,  0.30001694,
         -2.5025067 ,  0.48417033,  1.79246758],
        [-0.49105444,  0.59040511, -0.55467129,  0.2331845 ,  1.85863221,
         -1.52513589,  0.58108091, -0.71990732]]),
 array([[-1.

## Self Attention


$self attention = softmax \left({\dfrac{Q\cdot K^{T}}{\sqrt{d_k}}}+M\right)V$

In [5]:
scaled = np.matmul(Q, K.T) / math.sqrt(d_k)
scaled

array([[ 1.51392185, -1.2090864 ,  0.29168181, -0.58262083],
       [-0.83013406,  1.58292233,  1.39964973,  1.32511621],
       [-0.31087048,  1.18901265, -1.13432326, -0.75505218],
       [ 0.40369973, -0.9883311 ,  0.42180598, -0.41041243]])

In [14]:
# Masking is required in the decoder to ensure that we 
# don't get context from words generated in the future

mask = np.tril(np.ones((L, L)))
mask

array([[1., 0., 0., 0.],
       [1., 1., 0., 0.],
       [1., 1., 1., 0.],
       [1., 1., 1., 1.]])

In [ ]:
# transform the matrix such that 1 -> 0 and 0 -> -inf
# because the softmax is better if the values are in that range
mask[mask == 0] = -np.inf
mask[mask == 1] = 0
mask

array([[  0., -inf, -inf, -inf],
       [  0.,   0., -inf, -inf],
       [  0.,   0.,   0., -inf],
       [  0.,   0.,   0.,   0.]])

In [16]:
def softmax(x):
    return (np.exp(x).T / np.sum(np.exp(x), axis=-1)).T

In [24]:
self_attention = softmax(scaled + mask)

In [26]:
self_attention

array([[1.        , 0.        , 0.        , 0.        ],
       [0.08218248, 0.91781752, 0.        , 0.        ],
       [0.16891667, 0.75694352, 0.07413982, 0.        ],
       [0.3690209 , 0.09172752, 0.37576334, 0.16348825]])

In [28]:
new_V = np.matmul(self_attention, V)
new_V

array([[-1.79369791,  0.19252514,  0.50694942,  0.55103037, -1.66865031,
        -0.04663894, -0.02733588, -2.13418415],
       [-0.14625352,  0.23149883, -0.14499565,  1.10715724, -0.87766165,
        -0.11242727, -2.66223011,  1.26586166],
       [-0.37196368,  0.10767969, -0.03135365,  0.96809801, -0.81089291,
        -0.12123809, -2.14807552,  0.91566015],
       [-0.94674245, -0.21641519,  0.58670917,  0.37196797, -0.35244246,
        -0.4262432 , -0.17396348, -0.47783483]])

In [29]:
V

array([[-1.79369791e+00,  1.92525141e-01,  5.06949417e-01,
         5.51030372e-01, -1.66865031e+00, -4.66389433e-02,
        -2.73358778e-02, -2.13418415e+00],
       [ 1.26063007e-03,  2.34988581e-01, -2.03371596e-01,
         1.15695351e+00, -8.06835577e-01, -1.18318036e-01,
        -2.89816171e+00,  1.57030586e+00],
       [-9.43250757e-01, -1.38541074e+00,  4.98449447e-01,
        -9.82818756e-03,  1.10195704e+00, -3.21014084e-01,
         6.78268272e-01,  1.18056119e+00],
       [ 4.25064062e-01,  1.29410312e+00,  1.41288438e+00,
         4.04890522e-01, -4.69401699e-01, -1.69770072e+00,
        -9.35254569e-01, -1.69998860e+00]])